# Customer/Buyer Segmentation Based on Diamond Purchase Characteristics

The dataset has no customer IDs. Each row is therefore treated as an anonymous purchase profile. This notebook compares K-Means segmentations statistically, describes the diamond purchases inside each cluster, and only then infers a likely buyer orientation.


## 1. Configuration

All cleaning, feature engineering, scaling, encoding, evaluation, profiling, plots, and persistence live in `src/clustering/`. The default runs the complete dataset; set `SMOKE_TEST` to `True` for a quick rehearsal.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    folder for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "src" / "clustering" / "pipeline.py").is_file()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from IPython.display import Image, display
import pandas as pd

from src.clustering.pipeline import run_buyer_segmentation

SMOKE_TEST = False


## 2. Run K = 3, 5, 7, and 10

Every K uses the same preprocessed inputs and K-Means++ initialization. The terminal output reports silhouette, inertia, and the smallest cluster percentage as each candidate completes.


In [ ]:
result = run_buyer_segmentation(smoke=SMOKE_TEST)
result["comparison"]


## 3. Compare Statistical Evidence

The final K is not chosen from silhouette alone. The selection score combines separation, minimum centroid distance, cluster balance, minimum cluster size, and simplicity.


In [ ]:
output_root = PROJECT_ROOT / "outputs" / "clustering"
if SMOKE_TEST:
    output_root = output_root / "smoke_test"

display(Image(filename=output_root / "figures" / "k_comparison.png"))
result["selected_profiles"]


## 4. Selected Buyer-Segment Profiles

`Purchase_Profile` describes the observed diamonds. `Buyer_Interpretation` is a cautious inference about the likely buying orientation, not a claim about identified customers.


In [ ]:
selected = pd.read_csv(output_root / "tables" / "selected_buyer_segments.csv")
selected


## 5. Full Numeric Statistics

For every K and cluster, this table contains mean, median, minimum, maximum, range, and standard deviation.


In [ ]:
numeric_stats = pd.read_csv(output_root / "tables" / "numeric_cluster_statistics.csv")
numeric_stats.head(20)


## 6. Full Categorical Distributions

Counts and within-cluster percentages are exported for cut, color, and clarity.


In [ ]:
category_stats = pd.read_csv(output_root / "tables" / "categorical_cluster_distributions.csv")
category_stats.head(20)


## 7. Visualize the Selected Segmentation


In [ ]:
display(Image(filename=output_root / "figures" / "selected_clusters_pca.png"))
display(Image(filename=output_root / "figures" / "selected_cluster_sizes.png"))


## 8. Saved Model

The selected preprocessor, K-Means model, selected K, and profile labels are saved together under `models/clustering/`. Use `assign_purchase_segment` to assign a new raw diamond purchase to its nearest segment.


In [ ]:
from src.clustering.prediction import assign_purchase_segment, load_segmentation_model

model_root = PROJECT_ROOT / "models" / "clustering"
if SMOKE_TEST:
    model_root = model_root / "smoke_test"
artifact = load_segmentation_model(model_root / "buyer_segmentation.joblib")

example = {
    "price": 6000, "carat": 1.0, "cut": "Ideal", "color": "G", "clarity": "VS1",
    "depth": 61.5, "table": 57.0, "x": 6.45, "y": 6.43, "z": 3.96,
}
assign_purchase_segment(artifact, example)
